# ComfyUI on Colab → 外部公開セットアップ（Googleドライブの既存環境を使用）

このノートブックは、あなたのGoogleドライブのマイドライブに既にある `ComfyUI` フォルダ（モデル・LoRA・カスタムノード・ワークフロー一式が入ったもの）をそのままColab上で起動し、`cloudflared`で外部からアクセス可能な公開URLを発行します。
モデルの再ダウンロードは行いません。既存のマイドライブ内のファイルをそのまま使います。

発行されたURLを、手元の `app/index.html`（Comfy Simple Studio）の「バックエンドURL」欄に貼り付ければ使えます。

**前提**
- マイドライブ直下に `ComfyUI` フォルダが存在すること（`ComfyUI/models/checkpoints` などが入っている状態）
- ローカルPC側で同じComfyUIフォルダを同時に起動しない（Google Driveの同期やファイルロックが競合する可能性があるため）

**手順**
1. ランタイムタイプを GPU に変更（ランタイム → ランタイムのタイプを変更 → T4以上）
2. 上から順に全セルを実行
3. Googleドライブへのアクセス許可を求めるポップアップが出たら、自分のアカウントで許可する
4. 最後のセルに表示される `https://xxxx.trycloudflare.com` をコピー
5. Colabは一定時間操作がないと切断されます。切断されたら全セルを再実行してください（URLは毎回変わります）。

In [ ]:
# 1. Googleドライブをマウント
from google.colab import drive
drive.mount('/content/drive')

COMFY_DIR = '/content/drive/MyDrive/ComfyUI'
print('ComfyUIディレクトリ:', COMFY_DIR)

In [ ]:
# 2. ComfyUI本体の依存ライブラリをインストール
%cd {COMFY_DIR}
!pip install -r requirements.txt -q
print('ComfyUI本体の依存関係インストール完了')

In [ ]:
# 3. 使用するカスタムノードの依存ライブラリをインストール
# pose_variation_anima_V4.json ワークフローの解析結果、実際に使われているカスタムノードは以下の4つだけでした。
# 他のカスタムノード（Impact-pack, ControlNet aux, IPAdapter等）はこのワークフローでは未使用のため、
# pipインストールをスキップして起動時間を短縮します（フォルダ自体は残っているのでImportは試みられますが、
# 使わない機能なのでエラーが出ても無視して問題ありません）。
import os

USED_CUSTOM_NODES = [
    'comfyui-easy-use',        # easy imageListToImageBatch / easy imageCount
    'comfyui-custom-scripts',  # MathExpression (pysssss)
    'character_prompt_cycler', # LoadTextCycleFromFolder
    'ComfyUI-Anima-LLLite',    # AnimaLLLiteApply
]

for name in USED_CUSTOM_NODES:
    req = f'custom_nodes/{name}/requirements.txt'
    if os.path.exists(req):
        print('=== installing:', req, '===')
        !pip install -r "{req}" -q
    else:
        print(f'{name}: requirements.txt なし（インストール不要）')

print('使用するカスタムノードの依存関係インストール完了')
print('他のワークフローも使いたくなったら、そこで使われているカスタムノード名を USED_CUSTOM_NODES に追加してください。')

In [ ]:
# 4. cloudflared (トンネル用バイナリ) を取得
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print('cloudflared 準備完了')

In [ ]:
# 5. ComfyUI を起動し、cloudflared で外部公開 URL を発行
# --enable-cors-header は、別オリジン（ブラウザで開いている app/index.html）からの fetch を許可するために必須です。
import subprocess, threading, time, re

def stream_output(proc, prefix):
    for line in proc.stdout:
        print(prefix + line, end='')

import os as _os
_env = _os.environ.copy()
_env['PYTHONUNBUFFERED'] = '1'
comfy_proc = subprocess.Popen(
    ['python', '-u', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--enable-cors-header'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    cwd=COMFY_DIR,
    env=_env,
)
# ComfyUI自体のログをこのセルに流す（エラー時の原因調査に使う。生成が進まない/落ちた場合はここを確認）
threading.Thread(target=stream_output, args=(comfy_proc, '[ComfyUI] '), daemon=True).start()

# ComfyUI が起動するまで少し待つ（モデル数・カスタムノード数が多いので通常より時間がかかることがあります）
time.sleep(25)

tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

print('公開URLを取得中...')
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if m:
        print()
        print('==================================')
        print(' ComfyUI 公開URL:', m.group(0))
        print(' → app/index.html の「バックエンドURL」欄に貼り付けてください')
        print('==================================')
        print()
        break
# トンネルのログも引き続きこのセルに流す（切断検知用）
threading.Thread(target=stream_output, args=(tunnel_proc, '[cloudflared] '), daemon=True).start()
print('ComfyUI と cloudflared はバックグラウンドで実行中です。このセルの実行は終了しても問題ありません。')
print('生成が止まる/進まない場合は、このセルの出力に [ComfyUI] のログが増えているか確認してください。')

## トラブルシューティング
- Googleドライブのマウントで権限エラーになる: ポップアップで正しいGoogleアカウント（ComfyUIフォルダを持っているアカウント）を選んでいるか確認してください。
- `ComfyUI` フォルダが見つからないと出る: マイドライブ直下のフォルダ名が `ComfyUI` になっているか確認してください（大文字小文字も一致させる）。
- カスタムノードのインストールでエラーが出る: そのカスタムノードが原因でComfyUI自体は起動しないことがあります。エラーメッセージに出ているカスタムノード名を教えてください。
- URLが表示されない場合: 最後のセルをもう一度実行してください（トンネル接続に数秒かかることがあります）。
- アプリ側で「接続失敗」になる場合: URLの写し間違い(末尾の / など)を確認してください。
- 一定時間経つとColabセッションが切断されます。その場合は最初から全セルをやり直してください（URLが変わるのでアプリ側も更新）。
- ローカルPCで同じComfyUIフォルダを同時に開いていると、Google Driveの同期やモデルの読み込みで競合する可能性があります。Colabを使う間はローカル側のComfyUIは閉じておくことをおすすめします。